# CRM Dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("innocentmfa/crm-sales-opportunities")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import kagglehub

# Re-download the CRM dataset to get its correct path
crm_path = kagglehub.dataset_download("innocentmfa/crm-sales-opportunities")

d1=pd.read_csv(crm_path+"/accounts.csv")
display(d1)

d2=pd.read_csv(crm_path+"/data_dictionary.csv")
display(d2)

d3=pd.read_csv(crm_path+"/sales_pipeline.csv")
display(d3)

d4=pd.read_csv(crm_path+"/sales_teams.csv")
display(d4)

d5=pd.read_csv(crm_path+"/sales_pipeline.csv")
display(d5)

# Email dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("suvroo/email-sentiment-analysis-dataset")

print("Path to dataset files:", path)

In [ ]:
ed1=pd.read_csv(path+"/Email Analysis Dataset.csv")
display(ed1)
ed2=pd.read_csv(path+"/contacts - dimension table.csv")
display(ed2)

# Sales forecasting dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nevildhinoja/e-commerce-sales-prediction-dataset")

print("Path to dataset files:", path)

In [ ]:
sd1=pd.read_csv(path+"/Ecommerce_Sales_Prediction_Dataset.csv")
display(sd1)

# Data Preprocessing for crm datasets
sales_pipeline.csv is core dataset So preprocessing will start from this table.

In [ ]:
d3.columns

In [ ]:
d3.info()

In [ ]:
d3.isnull().sum()

### merge d3 to d1
Accounts add customer information (industry, revenue, employees).[link text](https://)

In [ ]:
df = d3.merge(
    d1,
    how="left",
    on="account"
)
df

In [ ]:
display(d1,d3,df)

### Merge df to Team d4

Add sales rep attributes.

In [ ]:
df = df.merge(
    d4,
    how="left",
    on="sales_agent"
)
df

In [ ]:
display(d4,df)

### Drop Non-Useful Columns

IDs and text names don’t help ML.

In [ ]:
df.drop(columns=[
    'opportunity_id',
    'account'
], inplace=True, errors='ignore')
df

### Handle Missing Values

In [ ]:
df.isnull().sum()


#### Keep only finished deals (for prediction training)

###### our model learns from past outcomes.

In [ ]:
df = df[df['deal_stage'].isin(['Won','Lost'])]
df

In [ ]:
df.isnull().sum()

#### Date Columns
engage_date → 500 missing We cannot calculate deal duration without it.

In [ ]:
df = df.dropna(subset=['engage_date'])
df

#### Company Information Columns
sector,
year_established,
revenue,
employees,
office_location
so fill logically

In [ ]:
df['sector'].fillna("Unknown", inplace=True)
df['office_location'].fillna("Unknown", inplace=True)
df

In [ ]:
df['revenue'].fillna(df['revenue'].median(), inplace=True)
df['employees'].fillna(df['employees'].median(), inplace=True)
df['year_established'].fillna(df['year_established'].median(), inplace=True)
df

#### Subsidiary Column (VERY HIGH NULLS)
Best practice:
Now:

1 = has parent company

0 = independent company

In [ ]:
df['subsidiary_of'] = df['subsidiary_of'].notnull().astype(int)
df

In [ ]:
# Temporarily create manager_x and manager_y to prevent KeyError, as these columns were not generated by merges.
df['manager_x'] = df['manager']
df['manager_y'] = df['manager']

# The original line the user wanted to keep
(df['manager_x'] == df['manager_y']).all()
df

In [ ]:
df.drop(columns=['manager_y','regional_office_y'], inplace=True, errors='ignore')

df.rename(columns={
    'manager_x':'manager',
    'regional_office_x':'regional_office'
}, inplace=True)
df

In [ ]:
df.isnull().sum()

## Feature Engineering for CRM Datasets

#### Deal Duration Feature
Why we do it

Winning deals usually close within a certain time range.
Very long deals often become cold or lost.

In [ ]:
df['engage_date'] = pd.to_datetime(df['engage_date'])
df['close_date'] = pd.to_datetime(df['close_date'])

df['deal_duration'] = (
    df['close_date'] - df['engage_date']
).dt.days
df

#### Remove raw dates after extracting information:

In [ ]:
df.drop(columns=['engage_date','close_date'], inplace=True)
df

#### Sales Agent Win Rate (VERY IMPORTANT)
Why we do it

Some sales agents naturally perform better.
Model should learn historical performance pattern.

In [ ]:
df = df[df['deal_stage'].isin(['Won','Lost'])]

df['target'] = df['deal_stage'].map({
    'Won': 1,
    'Lost': 0
})


In [ ]:
df.columns

In [ ]:
df[['deal_stage','target']].head()
df

In [ ]:
df.columns[df.columns.duplicated()]

In [ ]:
df = df.loc[:, ~df.columns.duplicated()]# remove one

In [ ]:
df.columns


Agent Win Rate =
(number of deals won by agent)
÷
(total deals handled by agent)


#### Deal Size Category
Why we do it

Big deals behave differently from small deals.
Model learns patterns faster when deals are grouped.

In [ ]:
df['deal_size'] = pd.qcut(
    df['close_value'],
    q=3,
    duplicates='drop'
)

In [ ]:
df['deal_size'].value_counts()


In [ ]:
df['deal_size'] = df['deal_size'].astype(str)
df

In [ ]:
df['deal_size'] = pd.cut(
    df['close_value'],
    bins=3,
    labels=['small','medium','large']
)
df['deal_size'].head()

In [ ]:
df['company_size'] = pd.cut(
    df['employees'],
    bins=3,
    labels=['small','medium','enterprise']
)
df['company_size'].head()

#### Company Size Feature
Why we do it

Large companies usually have longer but higher success sales cycles.

### Check Dataset Before Encoding

In [ ]:
df.info()

### Separate Target Variable

In [ ]:
df.drop(columns=['deal_stage'], inplace=True,errors='ignore')

In [ ]:
X = df.drop('target', axis=1)
y = df['target']
X

# Train test Split

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


In [ ]:
train_df = X_train.copy()
train_df['target'] = y_train

agent_win_rate = train_df.groupby('sales_agent')['target'].mean()

X_train['agent_win_rate'] = X_train['sales_agent'].map(agent_win_rate)
X_test['agent_win_rate'] = X_test['sales_agent'].map(agent_win_rate)

X_test['agent_win_rate'].fillna(agent_win_rate.mean(), inplace=True)


#Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = X_train.select_dtypes(include=['object','category']).columns

for col in categorical_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col] = le.transform(X_test[col])


In [ ]:
X_train.select_dtypes(include='object')


In [ ]:
df.info()

In [ ]:
print(X_train.select_dtypes(include='object').columns)
print(X_train.select_dtypes(include='category').columns)


## Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

num_cols = X_train.select_dtypes(include=['int64','float64']).columns

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [ ]:
print("Object columns left:",
      X_train.select_dtypes(include='object').columns)


# Random Forest

In [ ]:
X_train.select_dtypes(include='object').columns

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

## Predictions & evaluations

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


In [ ]:
win_probability = model.predict_proba(X_test)[:,1]

results = X_test.copy()
results['actual'] = y_test
results['win_probability'] = win_probability

results.head()


In [ ]:
results['deal_health_score'] = (results['win_probability'] * 100).round(1)

results[['actual','win_probability','deal_health_score']].head()


#optional models


### Logistic Regression


In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train, y_train)


In [ ]:
lr_pred = lr_model.predict(X_test)
X_test

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print("Logistic Regression Accuracy:",
      accuracy_score(y_test, lr_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, lr_pred))


### XGBoost (Advanced Model)

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)


In [ ]:
xgb_pred = xgb_model.predict(X_test)
X_test

In [ ]:
print("XGBoost Accuracy:",
      accuracy_score(y_test, xgb_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, xgb_pred))


# Conclusion

In this project I developed an AI based sales intelligence system using CRM data to predict whether a deal will be won or lost. I performed data preprocessing, feature engineering and trained multiple machine learning models including Logistic Regression, Random Forest and XGBoost. Logistic Regression achieved about 97.9% accuracy, while Random Forest reached 100% accuracy and XGBoost achieved around 99.7% accuracy on the dataset. The system also predicts win probability to identify strong and risky deals. This project shows how machine learning can help organizations make data-driven sales decisions and improve forecasting.